In [6]:
import os
from pathlib import Path
from typing import Optional

import fastmri
import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
from data_utils import *
from datasets import *
from fastmri.data.transforms import tensor_to_complex_np
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from torch.utils.data import DataLoader, TensorDataset
import re
from model import *
from torch.optim import SGD, Adam, AdamW
from train_utils import *
from itertools import chain
import copy 


In [3]:
vol = 0
n_volumes = 1
n_slices = 2
path_to_data = Path('/itet-stor/mcrespo/bmicdatasets-originals/Originals/fastMRI/brain/multicoil_train/')
# dataset = KCoordDataset(path_to_data, n_volumes=n_volumes, n_slices=2, with_mask=False, acceleration=4, center_frac=0.15)
# center = dataset.metadata[vol]["center"]
# left_idx, right_idx, center_vals = center["left_idx"], center["right_idx"], center["vals"]
# shape = dataset.metadata[vol]["shape"]
# n_slices, n_coils, height, width = shape

# Create tensors of indices for each dimension
# kx_ids = torch.cat([torch.arange(left_idx), torch.arange(right_idx, width)])
# ky_ids = torch.arange(height)
# kz_ids = torch.arange(n_slices)
# coil_ids = torch.arange(n_coils)

# Use meshgrid to create expanded grids
# kspace_ids = torch.meshgrid(kx_ids, ky_ids, kz_ids, coil_ids, indexing="ij")
# kspace_ids = torch.stack(kspace_ids, dim=-1).reshape(-1, len(kspace_ids))

path_to_data = Path(path_to_data)
kspace_gt = []
ground_truth = []
if path_to_data.is_dir():
    files = sorted(
        [
            file
            for file in path_to_data.iterdir()
            if file.suffix == ".h5" and "AXT1POST_205" in file.name            
        ]
    )[:n_volumes]
    
for file in files:
    with h5py.File(file, "r") as hf:
        volume_kspace = (to_tensor(preprocess_kspace(hf["kspace"][()]))[:n_slices])
        ground_truth.append(hf["reconstruction_rss"][()][: n_slices])


In [24]:

n_levels = 15

model_saved_path = '/scratch_net/ken/mcrespo/proj_marina/logs_new/train_hash/2025-01-20_00h34m04s/checkpoints/epoch_0599.pt'
checkpoint = torch.load(model_saved_path,  map_location=torch.device('cpu'))
model= Siren(coord_dim=4, levels=15, n_min=45, size_hashtable=12, hidden_dim=512, n_features=10, n_layers=8, out_dim=2, n_volumes=4)

levels = [[] for _ in range(n_levels)]  # Creates 8 empty lists

for layer_name, tensor in checkpoint['model_state_dict'].items():
    if 'embed_fn' in layer_name:
        numbers = re.findall(r'\d+', layer_name)
        # Convert the last number to an integer
        last_number = int(numbers[-1])
        
        # Assign tensor to the corresponding level if the number is valid
        if 0 <= last_number < len(levels):
            levels[last_number].append(tensor)

table_dict = []
for level in levels:
    meanoflevel = np.mean(level, axis = 0)
    table_dict.append(torch.tensor(meanoflevel))
    
for layer_name, tensor in model.named_parameters():
    if 'embed_fn' in layer_name:
        numbers = re.findall(r'\d+', layer_name)
        level = int(numbers[-1])
        # print(table_dict[level].shape)
        # print(tensor.shape)
        tensor.data.copy_(table_dict[level])
        # print(layer_name)
    else:
        tensor.data.copy_(checkpoint['model_state_dict'][layer_name])


for layer, tensor in model.named_parameters():
    if 'embed_fn' in layer:
        tensor.requires_grad = True
    else:
        tensor.requires_grad = False
    
### INSERT THE EMBEDDINGS TO BE OPTIMIZED INTO THE OPTIMIZER
optimizer = Adam(model.embed_fn.parameters(),
    lr=1.e-4,
)

model_copy = copy.deepcopy(model)

/tmp/ipykernel_13655/3105982219.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_saved_path,  map_location=torch.device('cpu'))


In [25]:
n_slices, n_coils, width, height = volume_kspace.shape[:-1]
vol_id = 0
limit = 4 # EVALUATED ON 4 POINTS - DISTANCED 1 pixel IN X DIMENSION
target = torch.zeros(limit, 2)
coords = torch.zeros(limit, 5)
inputs = torch.zeros(limit, 5)

scale = 10
for i in range(limit):
    posx = 50+i*scale
    posy = 120+i*scale
    posz = 0
    coil = 4
    posz_n = posz - 1
    coil_n = 2*(coil)/(n_coils - 1) - 1
    
    
    print(f'Pixel : {posx}, {posy}')
    ## ORDER IS: vol_id, x_coor, ycoor, z_coor, coil_id 
    inputs[i] = torch.tensor([0, posx, posy, posz_n, coil_n])
    ## ORDER IS: slice, coil_id, ycoor, xcoor 
    target[i]= volume_kspace[posz, coil, posy, posx, ...]
    coords[i] = inputs[i]
    
loss_fn = MSELoss(gamma=1)
optimizer.zero_grad(set_to_none=True)

for i in range(1):
    # Get the index for In the provided code snippet, the variable `t` is not explicitly defined or referenced. It seems to be missing or not relevant to the context provided. If you have more context or specific details about where `t` is used or defined in your code, please provide additional information so that I can assist you better.
    # optimizer.zero_grad(set_to_none=True)
    
    outputs = model(coords)
    loss = loss_fn(outputs, target)
    loss.backward()
    optimizer.step()

# for vol in range(4):
#     print(f'Volume {vol}')
vol = 0
for i, layer in enumerate(model_copy.embed_fn.vol_embeddings[vol]):
    trained_model = model.embed_fn.vol_embeddings[vol][i].weight.data
    buffer_model = layer.weight.data
    print(f'Level: {i} norm gradient change: {torch.norm(trained_model - buffer_model)}')

Pixel : 50, 120
Pixel : 60, 130
Pixel : 70, 140
Pixel : 80, 150
Level: 0 norm gradient change: 0.0011830073781311512
Level: 1 norm gradient change: 0.0012646970571950078
Level: 2 norm gradient change: 0.0012645893730223179
Level: 3 norm gradient change: 0.001264770980924368
Level: 4 norm gradient change: 0.0012647047406062484
Level: 5 norm gradient change: 0.0012647403636947274
Level: 6 norm gradient change: 0.0012642479268833995
Level: 7 norm gradient change: 0.00126481126062572
Level: 8 norm gradient change: 0.0012645409442484379
Level: 9 norm gradient change: 0.0012647989206016064
Level: 10 norm gradient change: 0.0012648036936298013
Level: 11 norm gradient change: 0.0012644700473174453
Level: 12 norm gradient change: 0.001264743972569704
Level: 13 norm gradient change: 0.0012647123076021671
Level: 14 norm gradient change: 0.0006323879933916032


In [76]:
for vol in range(4):
    print(f'Volume {vol}')
    for i, layer in enumerate(model_copy.embed_fn.vol_embeddings[vol]):
        trained_model = model.embed_fn.vol_embeddings[vol][i].weight.data
        buffer_model = layer.weight.data
        print(f'Level: {i} norm gradient change: {torch.norm(trained_model - buffer_model)}')

Volume 0
Level: 0 norm gradient change: 0.048485755920410156
Level: 1 norm gradient change: 0.05103255808353424
Level: 2 norm gradient change: 0.049541354179382324
Level: 3 norm gradient change: 0.049780283123254776
Level: 4 norm gradient change: 0.04985009878873825
Level: 5 norm gradient change: 0.0495225228369236
Level: 6 norm gradient change: 0.050891779363155365
Level: 7 norm gradient change: 0.05118440464138985
Level: 8 norm gradient change: 0.04899391159415245
Level: 9 norm gradient change: 0.05000463128089905
Level: 10 norm gradient change: 0.05026378110051155
Level: 11 norm gradient change: 0.052455998957157135
Level: 12 norm gradient change: 0.051341526210308075
Level: 13 norm gradient change: 0.04991351068019867
Level: 14 norm gradient change: 0.024676572531461716
Volume 1
Level: 0 norm gradient change: 0.0
Level: 1 norm gradient change: 0.0
Level: 2 norm gradient change: 0.0
Level: 3 norm gradient change: 0.0
Level: 4 norm gradient change: 0.0
Level: 5 norm gradient change: 

In [27]:
n_slices, n_coils, height, width = 2, 4, 320, 320
left_idx= 300
right_idx= 320
# vol_ids = torch.arange(5)
# Create tensors of indices for each dimension
kx_ids = torch.cat([torch.arange(left_idx), torch.arange(right_idx, width)])
ky_ids = torch.arange(height)
kz_ids = torch.arange(n_slices)
coil_ids = torch.arange(n_coils)

# Use meshgrid to create expanded grids
kspace_ids = torch.meshgrid(kx_ids, ky_ids, kz_ids, coil_ids, indexing="ij")
kspace_ids = torch.stack(kspace_ids, dim=-1).reshape(-1, len(kspace_ids))

dataset = TensorDataset(kspace_ids)
dataloader = DataLoader(
    dataset, batch_size=60_000, shuffle=False, num_workers=3
)

for vol_id in range(5):

    volume_kspace = torch.zeros(
        (n_slices, n_coils, height, width, 2),
        dtype=torch.float32,
    )
    print('Predicting volume')
    for point_ids in dataloader:
        point_ids = point_ids[0].long()
        coords = torch.zeros_like(
            point_ids, dtype=torch.float32
        )
        # Normalize the necessary coordinates for hash encoding to work
        coords[:, :2] = point_ids[:, :2]
        coords[:, 2] = (2 * point_ids[:, 2]) / (n_slices - 1) - 1
        coords[:, 3] = (2 * point_ids[:, 3]) / (n_coils - 1) - 1
        vol_id_vect = torch.ones(len(point_ids)).unsqueeze(1) * vol_id
        coords = torch.hstack([vol_id_vect, coords])
        

Predicting volume
Predicting volume
Predicting volume
Predicting volume
Predicting volume


In [ ]:
path_to_data = '/itet-stor/mcrespo/bmicdatasets-originals/Originals/fastMRI/brain/multicoil_train/'

dataset = KCoordDataset(path_to_data, n_volumes=n_volumes, n_slices=n_slices, with_mask=with_mask, acceleration=acceleration, center_frac=center_frac)
dataloader = DataLoader(
    dataset,
    batch_size=1_000,
    num_workers=0,
    shuffle=True,
)

vol_id = 0
shape = dataloader.dataset.metadata[vol_id]["shape"]
center_data = dataloader.dataset.metadata[vol_id]["center"]
left_idx, right_idx, center_vals = (
    center_data["left_idx"],
    center_data["right_idx"],
    center_data["vals"],
)
n_slices, n_coils, height, width = shape

volume_kspace = torch.zeros(
    (n_slices, n_coils, height, width, 2),
    dtype=torch.float32,
)
predicted_volume = volume_kspace.clone()

for batch_idx, (inputs,inputs_unnormalized,targets) in enumerate(dataloader):
    
    coords = inputs[:, 1:-1] # kx,ky,kz
    vol_ids = inputs[:,0].long()
    coil_ids = inputs[:,-1].long() # unnormalized coilID
    
    latent_vol = embedding_vol[vol_ids]
    latent_coil = embedding_coil[start_idx[vol_ids] + coil_ids]

    # outputs = model(coords, latent_vol, latent_coil)
    
    # predicted_volume[inputs_unnormalized[:,2], inputs_unnormalized[:,3], inputs_unnormalized[:,1], inputs_unnormalized[:,0]] = outputs
    volume_kspace[inputs_unnormalized[:,2], inputs_unnormalized[:,3], inputs_unnormalized[:,1], inputs_unnormalized[:,0]] = targets

In [ ]:
gt_modulus = np.abs(fft2_shift(gt_img))
predicted_modulus = np.abs(fft2_shift(predicted_img))


plt.figure(figsize=(15,10))
plt.subplot(1,2,1)
plt.imshow(np.log(gt_modulus[0] + eps))
plt.axis('off')
plt.title('Undersampled groundtruth')
plt.colorbar()
plt.subplot(1,2,2)
plt.imshow(np.log(predicted_modulus[0] + eps))
plt.axis('off')
plt.title('Undersampled predictions')
plt.colorbar()

In [ ]:
plt.figure(figsize=(10,10))

plt.subplot(2, 2, 1)
plt.imshow(np.log(gt_modulus[0] / dataloader.dataset.metadata[vol_id]["norm_cste"] + eps))
plt.axis('off')
plt.colorbar()
plt.title("Kspace")
plt.subplot(2, 2, 2)
plt.hist(np.log(gt_modulus[0].flatten()), log=True, bins=100)
plt.subplot(2, 2, 3)
plt.imshow(np.log(predicted_modulus[0] / dataloader.dataset.metadata[vol_id]["norm_cste"] + eps))
plt.axis('off')
plt.colorbar()
plt.title("Kspace")
plt.subplot(2, 2, 4)
plt.hist(np.log(predicted_modulus[0].flatten()), log=True, bins=100)
plt.show()


In [ ]:
vol_id = 0
file = dataloader.dataset.metadata[vol_id]["file"]
with h5py.File(file, "r") as hf:
    ground_truth = hf["reconstruction_rss"][()][: n_slices]
    
modulus = np.abs(fft2_shift(ground_truth))

